# DEH 30-Day PySpark Challenge
## Day 26 — Practice Set 5: Nulls & Data Quality

**Instructions:**
1. `File → Save a copy in Drive` first
2. Attempt each problem before checking the reference solution
3. Datasets are created inline below — deliberately messy

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Setup — messy orders dataset (used in Problems 1-4 and 6)

In [ ]:
messy_data = [
    ("O0001", "C001", "Electronics", 2,    1299.99, "2023-01-05"),
    ("O0002", "C002", "Electronics", 1,    None,    "2023-01-07"),
    ("O0003", None,   "Furniture",   4,    349.99,  "2023-01-10"),
    ("O0004", "C004", None,          -1,   89.99,   "2023-01-12"),
    ("O0005", "C005", "Electronics", None, 29.99,   None),
    (None,    "C006", "Furniture",   2,    -50.0,   "2023-01-18"),
    ("O0007", "C007", "Electronics", 1,    None,    "2023-01-20"),
    ("O0008", "C008", "Furniture",   0,    59.99,   "2023-01-22"),
    ("O0009", "C009", "Electronics", 1,    1299.99, "2023-01-25"),
    ("O0010", "C010", None,          2,    79.99,   "2023-01-28")
]

messy_schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("category",    StringType(), True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(), True),
    StructField("order_date",  StringType(), True)
])

messy_df = spark.createDataFrame(messy_data, messy_schema) \
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))

messy_df.show()

---
## Problem 1 (Easy) — Data quality audit

Produce a one-row summary showing null count per column.

In [ ]:
# Your attempt


### Reference Solution — Problem 1

In [ ]:
null_counts = messy_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in messy_df.columns
])
null_counts.show()

---
## Problem 2 (Easy) — Drop rows missing critical fields only

Drop where order_id OR customer_id is null. Keep rows with other nulls.

In [ ]:
# Your attempt


### Reference Solution — Problem 2

In [ ]:
cleaned = messy_df.dropna(subset=["order_id", "customer_id"])
print(f"Before: {messy_df.count()} | After: {cleaned.count()}")
cleaned.show()

---
## Problem 3 (Medium) — Smart fill using category average

Fill null unit_price with the average unit_price for that row's category.

In [ ]:
# Your attempt


### Reference Solution — Problem 3

In [ ]:
# Step 1 — compute average price per category (ignoring nulls automatically)
category_avg = messy_df.groupBy("category").agg(
    F.round(F.avg("unit_price"), 2).alias("category_avg_price")
)

# Step 2 — join back and coalesce
filled = messy_df.join(category_avg, on="category", how="left") \
    .withColumn(
        "unit_price_filled",
        F.coalesce(F.col("unit_price"), F.col("category_avg_price"))
    )

filled.select("order_id", "category", "unit_price", "category_avg_price", "unit_price_filled").show()

---
## Problem 4 (Medium) — Flag suspicious records

Flag 'suspicious' if quantity <= 0, OR unit_price < 0, OR order_date is null. Else 'valid'. Count both.

In [ ]:
# Your attempt


### Reference Solution — Problem 4

In [ ]:
flagged = messy_df.withColumn(
    "flag",
    F.when(
        (F.col("quantity") <= 0) | (F.col("unit_price") < 0) | (F.col("order_date").isNull()),
        "suspicious"
    ).otherwise("valid")
)

flagged.select("order_id", "quantity", "unit_price", "order_date", "flag").show()
flagged.groupBy("flag").count().show()

---
## Problem 5 (Hard) — Reconcile two data sources

Merge CRM and web signup customer records using coalesce, preferring CRM.

In [ ]:
crm_data = [
    ("C001", "James Anderson", None,              "Enterprise"),
    ("C002", None,              "310-555-0102",   None),
    ("C003", "Robert Johnson",  "312-555-0103",   "Enterprise")
]
web_data = [
    ("C001", "James A.",        "212-555-0101",   "Enterprise"),
    ("C002", "Maria Garcia",    None,              "SMB"),
    ("C003", "Robert Johnson",  None,              None)
]

crm_df = spark.createDataFrame(crm_data, ["customer_id", "name", "phone", "segment"])
web_df = spark.createDataFrame(web_data, ["customer_id", "name", "phone", "segment"])

print("CRM source:")
crm_df.show()
print("Web source:")
web_df.show()

In [ ]:
# Your attempt


### Reference Solution — Problem 5

In [ ]:
crm = crm_df.alias("crm")
web = web_df.alias("web")

merged = crm.join(web, on="customer_id", how="outer") \
    .select(
        F.col("customer_id"),
        F.coalesce(F.col("crm.name"),    F.col("web.name")).alias("name"),
        F.coalesce(F.col("crm.phone"),   F.col("web.phone")).alias("phone"),
        F.coalesce(F.col("crm.segment"), F.col("web.segment")).alias("segment")
    )

merged.show()

---
## Problem 6 (Hard) — Data quality scorecard

Completeness % per column = (non-null count / total) * 100, rounded 1 decimal. Sort ascending.

In [ ]:
# Your attempt


### Reference Solution — Problem 6

The trick: compute per-column null counts as before, then pivot the result from wide (one row, many columns) to long (one row per column) using `stack()` or by unioning individual selects.

In [ ]:
total = messy_df.count()

scorecard_rows = []
for c in messy_df.columns:
    non_null_count = messy_df.filter(F.col(c).isNotNull()).count()
    completeness = round(100.0 * non_null_count / total, 1)
    scorecard_rows.append((c, completeness))

scorecard_df = spark.createDataFrame(scorecard_rows, ["column_name", "completeness_pct"])
scorecard_df.orderBy("completeness_pct").show()

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*